# RAG System Evaluation Analysis

This notebook provides analysis tools for RAG evaluation results.

In [ ]:
# Import dependencies
import json
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

## Load Evaluation Results

In [ ]:
# Load evaluation results
def load_evaluation_results(results_path: str) -> dict:
    """Load evaluation results from JSON file."""
    with open(results_path) as f:
        return json.load(f)

# Find the latest evaluation file
eval_dir = Path('../data/evaluation_results')
eval_files = list(eval_dir.glob('eval_*.json'))

if eval_files:
    latest_file = max(eval_files, key=lambda x: x.stat().st_mtime)
    results = load_evaluation_results(latest_file)
    print(f"Loaded: {latest_file}")
else:
    print("No evaluation results found. Run evaluate_retrieval.py first.")
    results = None

## Summary Statistics

In [ ]:
if results:
    # Create summary dataframe
    summary_data = {
        'Metric': [],
        'Average': [],
        'Std Dev': [],
        'Min': [],
        'Max': []
    }
    
    for metric, avg in results['avg_scores'].items():
        summary_data['Metric'].append(metric)
        summary_data['Average'].append(avg)
        summary_data['Std Dev'].append(results['std_scores'].get(metric, 0))
        summary_data['Min'].append(results['min_scores'].get(metric, 0))
        summary_data['Max'].append(results['max_scores'].get(metric, 0))
    
    summary_df = pd.DataFrame(summary_data)
    display(summary_df.round(3))

## Metric Distribution Visualization

In [ ]:
if results:
    # Extract per-query scores
    per_query_data = []
    for r in results['results']:
        for metric_name, metric_data in r['metrics'].items():
            per_query_data.append({
                'query_id': r['sample_id'],
                'query': r['query'][:50] + '...' if len(r['query']) > 50 else r['query'],
                'metric': metric_name,
                'score': metric_data['score']
            })
    
    df = pd.DataFrame(per_query_data)
    
    # Box plot
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.boxplot(data=df, x='metric', y='score', ax=ax)
    ax.set_title('Distribution of Evaluation Metrics')
    ax.set_ylim(0, 1)
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.show()

## Per-Query Analysis

In [ ]:
if results:
    # Create per-query dataframe
    query_df = pd.DataFrame([
        {
            'Query': r['query'][:40] + '...' if len(r['query']) > 40 else r['query'],
            'Avg Score': r['avg_score'],
            **{f"{k}_score": v['score'] for k, v in r['metrics'].items()}
        }
        for r in results['results']
    ])
    
    # Heatmap
    fig, ax = plt.subplots(figsize=(12, 6))
    score_cols = [c for c in query_df.columns if c.endswith('_score')]
    heatmap_data = query_df[score_cols].values
    
    sns.heatmap(
        heatmap_data,
        xticklabels=[c.replace('_score', '') for c in score_cols],
        yticklabels=query_df['Query'].values,
        annot=True,
        fmt='.2f',
        cmap='RdYlGn',
        vmin=0,
        vmax=1,
        ax=ax
    )
    ax.set_title('Per-Query Evaluation Scores')
    plt.tight_layout()
    plt.show()

## Correlation Analysis

In [ ]:
if results:
    # Correlation between metrics
    score_cols = [c for c in query_df.columns if c.endswith('_score')]
    correlation_matrix = query_df[score_cols].corr()
    
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(
        correlation_matrix,
        annot=True,
        fmt='.2f',
        cmap='coolwarm',
        center=0,
        ax=ax
    )
    ax.set_title('Correlation Between Evaluation Metrics')
    plt.tight_layout()
    plt.show()

## Identify Problematic Queries

In [ ]:
if results:
    # Find queries with low scores
    low_threshold = 0.7
    problematic = query_df[query_df['Avg Score'] < low_threshold].sort_values('Avg Score')
    
    if len(problematic) > 0:
        print(f"Queries with average score below {low_threshold}:")
        display(problematic)
    else:
        print(f"No queries with average score below {low_threshold}")

## Hyperparameter Tuning Results

In [ ]:
# Load tuning results
tuning_file = eval_dir / 'tuning_results.json'

if tuning_file.exists():
    with open(tuning_file) as f:
        tuning_results = json.load(f)
    
    # Create tuning dataframe
    tuning_df = pd.DataFrame(tuning_results['all_results'])
    
    # Best configuration
    print("Best Configuration:")
    print(json.dumps(tuning_results['best_config'], indent=2))
    
    # Visualize
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # By chunk size
    tuning_df.groupby('chunk_size')['overall_score'].mean().plot(
        kind='bar', ax=axes[0], title='Score by Chunk Size'
    )
    
    # By overlap
    tuning_df.groupby('chunk_overlap')['overall_score'].mean().plot(
        kind='bar', ax=axes[1], title='Score by Chunk Overlap'
    )
    
    # By top-k
    tuning_df.groupby('top_k')['overall_score'].mean().plot(
        kind='bar', ax=axes[2], title='Score by Top-K'
    )
    
    for ax in axes:
        ax.set_ylim(0, 1)
        ax.set_xlabel('')
    
    plt.tight_layout()
    plt.show()
else:
    print("No tuning results found. Run tune_hyperparameters.py first.")

## Recommendations

In [ ]:
if results:
    print("\n=== RECOMMENDATIONS ===\n")
    
    # Based on metric analysis
    avg_scores = results['avg_scores']
    
    if avg_scores.get('faithfulness', 1) < 0.8:
        print("⚠️ Low faithfulness detected:")
        print("   - Consider increasing top-k to provide more context")
        print("   - Review prompt template for grounding instructions")
    
    if avg_scores.get('answer_relevancy', 1) < 0.8:
        print("⚠️ Low answer relevancy detected:")
        print("   - Review prompt template for answer quality")
        print("   - Consider using a more capable LLM")
    
    if avg_scores.get('context_relevancy', 1) < 0.8:
        print("⚠️ Low context relevancy detected:")
        print("   - Consider adjusting chunk size")
        print("   - Add reranking step for retrieved documents")
        print("   - Improve document quality and relevance")
    
    if all(s >= 0.8 for s in avg_scores.values()):
        print("✅ All metrics are performing well!")
        print("   - Consider testing on more diverse queries")
        print("   - Monitor performance in production")